In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import numpy as np

In [2]:
bus = pd.read_csv('Nodes/N-Bus.csv')
fgc = pd.read_csv('Nodes/N-FGC.csv')
metro = pd.read_csv('Nodes/N-Metro.csv')
tram = pd.read_csv('Nodes/N-Tram.csv')

all_stops = pd.concat([ bus, fgc, metro, tram], ignore_index=True)

exchange_edges = pd.read_csv('Edges/Exchanges-Reduced.csv')
exchange_edges = exchange_edges[~exchange_edges['lines'].str.contains('L12')] # eliminar si reexecutem l'script dels exchanges


In [3]:
exchange_edges = exchange_edges.merge(all_stops[['id','stop_id']], left_on='dest', right_on='id', how='left')

# Tram

In [4]:
exchange_edges_tram = exchange_edges[exchange_edges['dest'].str.startswith('T')]
print(len(exchange_edges_tram))

1596


In [5]:
stop_times_x = pd.read_table('Data/Tram/TBX/stop_times.txt', sep=',')
stop_times_S = pd.read_table('Data/Tram/TBS/stop_times.txt', sep=',')
stop_times = pd.concat([stop_times_x, stop_times_S])    
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [6]:
stops_x = pd.read_table('Data/Tram/TBX/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops_s = pd.read_table('Data/Tram/TBS/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops = pd.concat([stops_x, stops_s])
stops = stops[~stops['stop_id'].str.startswith('S')]
stops['stop_id'] = stops['stop_id'].astype(int)

In [7]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [8]:
trips_x = pd.read_table('Data/Tram/TBX/trips.txt', sep=',')
trips_s = pd.read_table('Data/Tram/TBS/trips.txt', sep=',')
trips = pd.concat([trips_x, trips_s])
trips = trips[['route_id','trip_id']]

In [9]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','route_id','departure_time'], keep='first', inplace=True)
times_w_routes['route_id'] = 'T' + times_w_routes['route_id'].astype(int).astype(str)
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].str[2:]
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].replace({'RIGL':'SRMN','LLEV':'BDOR','MRSM':'DGMR','CATA':'CTLN','JOAN':'STJB','MRTI':'STMR','SROC':'STRC'})
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
)
times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'stop_name', 'stop_desc'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'stop_name', 'stop_desc'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'stop_name', 'stop_desc'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])
avg_wait_times['id'] = 'T' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_desc']
avg_wait_times = avg_wait_times[['id', 'wait_time']]
avg_wait_times

,id,wait_time
0,T-T1-XILE,5.534852
1,T-T1-BVIA,3.198554
2,T-T1-OLIV,5.663828
3,T-T1-CLOT,5.663828
4,T-T1-SRMN,5.569308
...,...,...
81,T-T6-SABS,4.420414
82,T-T6-MINA,8.245186
83,T-T6-PARC,8.245186
84,T-T6-PORT,8.245186


In [10]:
exchange_edges_tram = exchange_edges_tram.merge(avg_wait_times, left_on='dest', right_on='id', how='left')
exchange_edges_tram = exchange_edges_tram[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]
len(exchange_edges_tram)

1596

# FCG

In [12]:
exchange_edges_fgc = exchange_edges[exchange_edges['dest'].str.startswith('F')]
len(exchange_edges_fgc)

300

In [13]:
stop_times = pd.read_table('Data/FGC/gtfs_fgc/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [14]:
stops = pd.read_table('Data/FGC/gtfs_fgc/stops.txt', sep=',')[['stop_id','stop_name']]

In [15]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [16]:
trips = pd.read_table('Data/FGC/gtfs_fgc/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
trips = trips[['route_id','trip_id','trip_headsign']]
valid = ['L6', 'L7', 'L8', 'L12','S1']
trips = trips[trips['route_id'].isin(valid)]

In [17]:
times_w_routes = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['route_id','stop_name','departure_time','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['departure_time'].notna()]
times_w_routes['stop_id']  = times_w_routes['stop_id'].str[:2]
times_w_routes

,route_id,trip_id,trip_headsign,stop_id,stop_name,departure_time
8,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,MN,Muntaner,07:00:00
9,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,BN,La Bonanova,07:01:00
10,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,TT,Les Tres Torres,07:03:00
11,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,SR,Sarrià,07:04:00
13,L6,6c4bdae602747613ef|6f2dc7e203,Sarrià,PC,Barcelona - Plaça Catalunya,07:07:00
...,...,...,...,...,...,...
29661,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,AL,Almeda,08:47:15
29662,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,CO,Cornellà Riera,08:45:15
29663,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,BO,Sant Boi,08:42:00
29664,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,EU,Europa | Fira,08:55:00


In [18]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id','trip_headsign'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
)
 

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'stop_id', 'stop_name'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'stop_id', 'stop_name'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])



avg_wait_times = avg_wait_times.groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['id'] = 'F' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,stop_id,stop_name,wait_time,id
0,L12,RE,Reina Elisenda,2.042273,F-L12-RE
1,L12,SR,Sarrià,2.042273,F-L12-SR
2,L6,BN,La Bonanova,2.368944,F-L6-BN
3,L6,GR,Gràcia,1.924447,F-L6-GR
4,L6,MN,Muntaner,2.367738,F-L6-MN
5,L6,PC,Barcelona - Plaça Catalunya,1.923921,F-L6-PC
6,L6,PR,Provença,1.908047,F-L6-PR
7,L6,SG,Sant Gervasi,2.369616,F-L6-SG
8,L6,SR,Sarrià,2.380794,F-L6-SR
9,L6,TT,Les Tres Torres,2.377072,F-L6-TT


In [ ]:
exchange_edges_fgc = exchange_edges_fgc.merge(avg_wait_times[['id','wait_time']], left_on='dest', right_on='id', how='left')
exchange_edges_fgc = exchange_edges_fgc[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry
0,SF-TB,F-L7-TB,Av. Tibidabo - Av. Tibidabo,FGC - FGC,IU Stop - L7,Exchange - Self,3.000000,2.002982,True,POINT (2.137162791 41.40964799000001)
1,SF-VL,F-S1-VL,Baixador de Vallvidrera - Baixador de Vallvidrera,FGC - FGC,IU Stop - S1,Exchange - Self,3.000000,3.430635,True,POINT (2.096936777 41.420094410000004)
2,SF-PC,F-L7-PC,Catalunya - Catalunya,FGC - FGC,IU Stop - L7,Exchange - Self,3.000000,2.048384,True,POINT (2.168720218999999 41.38563194)
3,F-S1-PC,F-L7-PC,Catalunya - Catalunya,FGC - FGC,S1 - L7,Exchange - Self,2.000000,2.048384,True,POINT (2.168720218999999 41.38563194)
4,F-L6-PC,F-L7-PC,Catalunya - Catalunya,FGC - FGC,L6 - L7,Exchange - Self,2.000000,2.048384,True,POINT (2.168720218999999 41.38563194)
...,...,...,...,...,...,...,...,...,...,...
295,B-V7-134,F-S1-SR,Pg Bonanova - Sarrià,Bus - FGC,V7 - S1,Exchange,4.466667,3.454893,True,"LINESTRING (2.123219 41.4007346, 2.1238758 41...."
296,B-68-622,F-S1-SR,Pg Bonanova - Sarrià,Bus - FGC,68 - S1,Exchange,4.933333,3.454893,True,"LINESTRING (2.1228245 41.4004882, 2.1234503 41..."
297,B-V7-622,F-S1-SR,Pg Bonanova - Sarrià,Bus - FGC,V7 - S1,Exchange,4.933333,3.454893,True,"LINESTRING (2.1228245 41.4004882, 2.1234503 41..."
298,B-68-1573,F-S1-SR,Via Augusta - Els Vergós - Sarrià,Bus - FGC,68 - S1,Exchange,2.433333,3.454893,True,"LINESTRING (2.1274552 41.3974569, 2.1274787 41..."


# Bus

In [20]:
exchange_edges_bus = exchange_edges[exchange_edges['dest'].str.startswith('B')]
len(exchange_edges_bus)

111662

## AMB

In [21]:
stop_times = pd.read_table('Data/GTFS_AMB/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id', 'stop_id', 'departure_time']].dropna()

stop_times['departure_time'] = pd.to_datetime(
    stop_times['departure_time'],
    format='%H:%M:%S',
    errors='coerce'
)

stop_times = stop_times[stop_times['departure_time'].dt.hour.between(7, 11)]
stop_times

,trip_id,stop_id,departure_time
0,129.20.1.1.0,107229,1900-01-01 08:55:00
1,129.20.1.1.0,2234,1900-01-01 08:56:09
2,129.20.1.1.0,2235,1900-01-01 08:57:46
3,129.20.1.1.0,476,1900-01-01 08:58:45
4,129.20.1.1.0,365,1900-01-01 08:59:57
...,...,...,...
890865,516.8.2.4.23,109921,1900-01-01 11:53:39
890866,516.8.2.4.23,106867,1900-01-01 11:54:41
890867,516.8.2.4.23,106868,1900-01-01 11:55:16
890868,516.8.2.4.23,106869,1900-01-01 11:57:03


In [22]:
stops = pd.read_table('Data/GTFS_AMB/stops.txt', sep=',')[['stop_id','stop_name']]

In [23]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [24]:
trips = pd.read_table('Data/GTFS_AMB/trips.txt', sep=',')
trips = trips[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_AMB/routes.txt', sep=',')[['route_id','route_short_name']] 
trips = trips.merge(routes, on='route_id', how='left')

In [25]:
times_w_routes_amb = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes_amb = times_w_routes_amb[times_w_routes_amb['departure_time'].notna()]
times_w_routes_amb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_amb['stop_id']  =  times_w_routes_amb['stop_id'].astype(int).astype(str)

## TMB

In [ ]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]
stop_times['stop_id'] = stop_times['stop_id'].astype(str)
stop_times = stop_times[stop_times['trip_id'].str.startswith('2')]

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/3576717721.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


,trip_id,stop_id,departure_time
777221,2.1.121.3345302.4094,2.9678.700773,10:05:00
777222,2.1.121.3345302.4094,2.9680.700539,10:10:00
777223,2.1.121.3345302.4094,2.8319.700682,10:19:00
777224,2.1.121.3345302.4094,2.1846.700880,10:21:00
777225,2.1.121.3345302.4094,2.9937.700777,10:24:00
...,...,...,...
1827645,2.250.102.3185372.3051,2.1595.694767,11:08:00
1827646,2.250.102.3185373.3051,2.1595.694767,11:18:00
1827651,2.250.102.3185373.3051,2.1622.691609,11:26:00
1827660,2.250.102.3185373.3051,2.819.695633,11:39:00


In [27]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops['stop_id'] = stops['stop_id'].astype(str)

In [28]:
stops_w_times = stop_times.merge(stops, on='stop_id', how='left')

In [29]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5','L9','L10','L11','L9N','L9S','L10N','L10S','FM','M1','M9','978']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[~trips['route_short_name'].isin(metro)]

short routes, routes that don't represent real lines (ex trip Id = 2.239.105.3380438.4092)

In [30]:
times_w_routes_tmb = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes_tmb = times_w_routes_tmb[times_w_routes_tmb['route_id'].notna()]
times_w_routes_tmb['stop_id'] = times_w_routes_tmb['stop_id'].str.split('.').str[1]
times_w_routes_tmb

,trip_id,stop_id,departure_time,stop_name,route_id,route_short_name,trip_headsign
1404,2.6.57.3060348.2899,3257,09:00:00,Llacuna - Ramon Turró - Final de trajecte,2.6.2899,6,Pg. Manuel Girona
1405,2.6.57.3060348.2899,3400,09:07:00,Marina - Meridiana,2.6.2899,6,Pg. Manuel Girona
1406,2.6.57.3060348.2899,317,09:15:00,Pl Tetuan - Pg de Sant Joan,2.6.2899,6,Pg. Manuel Girona
1407,2.6.57.3060348.2899,1520,09:25:00,Diagonal - Pg de Gràcia,2.6.2899,6,Pg. Manuel Girona
1408,2.6.57.3060348.2899,3932,09:42:00,Pg Manuel Girona - Benet Mateu - Final de traj...,2.6.2899,6,Pg. Manuel Girona
...,...,...,...,...,...,...,...
68802,2.250.102.3185372.3051,1595,11:08:00,Centre Esportiu Ciutat Meridiana - Final trajecte,2.250.3051,D50,Ciutat Meridiana
68803,2.250.102.3185373.3051,1595,11:18:00,Centre Esportiu Ciutat Meridiana - Final trajecte,2.250.3051,D50,Paral·lel
68804,2.250.102.3185373.3051,1622,11:26:00,S'Agaró - Sa Tuna,2.250.3051,D50,Paral·lel
68805,2.250.102.3185373.3051,819,11:39:00,Pg Verdum - Pl de la República - Llucmajor,2.250.3051,D50,Paral·lel


## Together

In [33]:
times_w_routes = pd.concat([times_w_routes_tmb, times_w_routes_amb], ignore_index=True)

In [34]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'trip_headsign','departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id','trip_headsign'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'route_short_name', 'stop_id', 'stop_name'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])

avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'B' + '-' + avg_wait_times['route_short_name'] + '-' + avg_wait_times['stop_id']

In [ ]:
exchange_edges_bus = exchange_edges_bus.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_bus = exchange_edges_bus[['origen', 'dest', 'tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry
0,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
1,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
2,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
3,B-V27-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
4,B-59-2,B-V27-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - V27,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
...,...,...,...,...,...,...,...,...,...,...
107745,B-B23-100560,B-B23-107806,Els Safaretjos - Pg. Pollancreda,Bus - Bus,B23 - B23,Exchange,4.95,NaN,True,"LINESTRING (2.2131607 41.4354834, 2.2138345 41..."
107746,B-B20-100560,B-B23-107806,Els Safaretjos - Pg. Pollancreda,Bus - Bus,B20 - B23,Exchange,4.95,NaN,True,"LINESTRING (2.2131607 41.4354834, 2.2138345 41..."
107747,B-B14-102045,B-B23-107806,Ctra. de Sta. Coloma - Madrid - Pg. Pollancreda,Bus - Bus,B14 - B23,Exchange,4.15,NaN,True,"LINESTRING (2.2142065 41.43565, 2.2143505 41.4..."
107748,B-B21-102045,B-B23-107806,Ctra. de Sta. Coloma - Madrid - Pg. Pollancreda,Bus - Bus,B21 - B23,Exchange,4.15,NaN,True,"LINESTRING (2.2142065 41.43565, 2.2143505 41.4..."


In [36]:
na_buses = exchange_edges_bus[exchange_edges_bus['wait_time'].isna()]
na_buses['line'] = exchange_edges_bus['lines'].str.split(' - ').str[1]
na_buses['dest_stop'] = exchange_edges_bus['dest'].str.split('-').str[2]
exchange_edges_bus = exchange_edges_bus[exchange_edges_bus['wait_time'].notna()]
na_buses['line'] = na_buses['lines'].str.split(' - ').str[1]

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/1131190627.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  na_buses['line'] = exchange_edges_bus['lines'].str.split(' - ').str[1]
/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/1131190627.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  na_buses['dest_stop'] = exchange_edges_bus['dest'].str.split('-').str[2]


In [ ]:
avg_wait_times_line = avg_wait_times.groupby(['route_short_name'], as_index=False)['wait_time'].mean().rename(columns={'wait_time': 'avg_wait_time'})
na_buses = na_buses.merge(avg_wait_times_line, left_on='line', right_on='route_short_name', how='left')
na_buses['wait_time'] = na_buses['avg_wait_time']
no_longer_na_buses  = na_buses[na_buses['wait_time'].notna()]
no_longer_na_buses.drop(columns=['line','dest_stop','avg_wait_time','route_short_name'], inplace=True)
exchange_edges_bus = pd.concat([exchange_edges_bus, no_longer_na_buses], ignore_index=True)

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry
0,B-H10-3,B-63-3,València - Muntaner - València - Muntaner,Bus - Bus,H10 - 63,Exchange - Self,1.00,4.852754,True,POINT (2.1578909990979396 41.388546000808674)
1,B-52-7,B-H12-7,Pl Universitat - Pl Universitat,Bus - Bus,52 - H12,Exchange - Self,1.00,2.337548,True,POINT (2.163310002198641 41.385906997198724)
2,B-H12-7,B-52-7,Pl Universitat - Pl Universitat,Bus - Bus,H12 - 52,Exchange - Self,1.00,20.901950,True,POINT (2.163310002198641 41.385906997198724)
3,B-7-21,B-M14-21,Palau Reial - Palau Reial,Bus - Bus,7 - M14,Exchange - Self,1.00,2.096133,True,POINT (2.116137003046015 41.38559299631731)
4,B-33-21,B-M14-21,Palau Reial - Palau Reial,Bus - Bus,33 - M14,Exchange - Self,1.00,2.096133,True,POINT (2.116137003046015 41.38559299631731)
...,...,...,...,...,...,...,...,...,...,...
111657,B-B23-100560,B-B23-107806,Els Safaretjos - Pg. Pollancreda,Bus - Bus,B23 - B23,Exchange,4.95,6.361241,True,"LINESTRING (2.2131607 41.4354834, 2.2138345 41..."
111658,B-B20-100560,B-B23-107806,Els Safaretjos - Pg. Pollancreda,Bus - Bus,B20 - B23,Exchange,4.95,6.361241,True,"LINESTRING (2.2131607 41.4354834, 2.2138345 41..."
111659,B-B14-102045,B-B23-107806,Ctra. de Sta. Coloma - Madrid - Pg. Pollancreda,Bus - Bus,B14 - B23,Exchange,4.15,6.361241,True,"LINESTRING (2.2142065 41.43565, 2.2143505 41.4..."
111660,B-B21-102045,B-B23-107806,Ctra. de Sta. Coloma - Madrid - Pg. Pollancreda,Bus - Bus,B21 - B23,Exchange,4.15,6.361241,True,"LINESTRING (2.2142065 41.43565, 2.2143505 41.4..."


# Metro

In [39]:
exchange_edges_metro = exchange_edges[exchange_edges['dest'].str.startswith('M')]

In [40]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/2260508506.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


In [41]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]

In [42]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [43]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[trips['route_short_name'].isin(metro)]

In [44]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','departure_time','route_id','route_short_name','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['route_id'].notna()]

In [45]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','route_short_name', 'trip_headsign','departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id','route_short_name'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'route_short_name', 'stop_id', 'stop_name'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])

avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'M' + '-' + avg_wait_times['route_short_name'] + '-' + avg_wait_times['stop_id']

In [46]:
exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
exchange_edges_metro['line_dest'] = exchange_edges_metro['lines'].str.split(' - ').str[-1]


/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/4268012821.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_94545/4268012821.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exchange_edges_metro['line_dest'] = exchange_edges_metro['lines'].str.split(' - ').str[-1]


In [ ]:
exchange_edges_metro = exchange_edges_metro.merge(avg_wait_times[['stop_name','wait_time','route_short_name']], left_on=['dest_name','line_dest'], right_on=['stop_name','route_short_name'], how='left')
exchange_edges_metro = exchange_edges_metro[['origen', 'dest','tram', 'mode','lines','type','time', 'wait_time','directed','geometry']]

516


516

# Save

In [49]:
transfer_weight = 8
w_weight = 1.75

In [50]:
print(len(exchange_edges))
exchanges = pd.concat([exchange_edges_tram, exchange_edges_fgc, exchange_edges_bus, exchange_edges_metro], ignore_index=True)
exchanges.insert(9, 'cost', exchanges['time'] + (exchanges['wait_time'] * w_weight) + transfer_weight)
print(len(exchanges))

114074
114074


In [51]:
exchanges.to_csv('Edges/Exchanges_with_Wait_Times.csv', index=False)